In [4]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 30))

In [6]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
df = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)

df

FETCHING DELIVERY BASKETS...: 100%|██████████| 6/6 [00:01<00:00,  3.46it/s]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,risk
0,1570142946000000101,2025-12-29 12:03:13+00:00,2026-08-06,2036-08-06 00:00:00,OIS_SWAP,10.147222,10Y,True,0.611111,7M,...,I,None,False,NaN,NaN,NaN,2036-08-06,NaN,NaN,2500.0
1,1570299802000000201,2025-12-29 12:05:58+00:00,2026-03-18,2046-03-18 00:00:00,OIS_SWAP,20.294444,IMM_H2046,True,0.219444,IMM_H2026,...,I,None,False,NaN,NaN,NaN,2046-03-18,NaN,NaN,2500.0
2,1570469108000000701,2025-12-29 12:09:34+00:00,2026-03-18,2051-03-18 00:00:00,OIS_SWAP,25.369444,IMM_H2051,True,0.219444,IMM_H2026,...,I,None,False,NaN,NaN,NaN,2051-03-18,NaN,NaN,2500.0
3,1570184451000000301,2025-12-29 12:09:46+00:00,2026-03-18,2031-03-18 00:00:00,OIS_SWAP,5.072222,IMM_H2031,True,0.219444,IMM_H2026,...,I,None,False,NaN,NaN,NaN,2031-03-18,NaN,NaN,17500.0
4,1570196242000000201,2025-12-29 12:11:05+00:00,2026-03-18,2036-03-18 00:00:00,OIS_SWAP,10.147222,IMM_H2036,True,0.219444,IMM_H2026,...,I,None,False,NaN,NaN,NaN,2036-03-18,NaN,NaN,10000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1605,1573649665000000201,2025-12-29 21:53:50+00:00,2025-12-31,2029-12-31 00:00:00,OIS_SWAP,4.058333,4Y,False,0.005556,spot,...,I,[1573649665000000201],True,91282CMD0,5-Year,2024-12-31,2029-12-31,low,NaN,2500.0
1606,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,I,[1573656731000000101],True,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,5000.0
1607,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,I,[1573691775000000101],True,91282CPQ8,7-Year,2025-12-31,2032-12-31,low,NaN,2500.0
1608,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,I,[1573723141000000201],True,912810UP1,30-Year,2025-11-17,2055-11-15,high,NaN,100000.0


In [14]:
# df[df["invoice_swap_ticker"].notna()]
df[(df["invoice_swap_ticker"].isna()) & (df["matched_ust_maturity_trade_confidence"] == "high") & (df["effective_date"].dt.date == datetime.date(2026, 3, 31))]

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,risk
974,1570380985000000401,2025-12-29 12:38:22+00:00,2026-03-31,2026-12-31 00:00:00,OIS_SWAP,0.763889,9M,True,0.255556,3M,...,I,[1570380985000000401],True,91282CME8,2-Year,2024-12-31,2026-12-31,high,NaN,5000.0
1600,1573630150000000401,2025-12-29 21:40:02+00:00,2026-03-31,2044-05-15 00:00:00,OIS_SWAP,18.391667,18Y,True,0.255556,3M,...,I,[1573630150000000401],True,912810UB2,20-Year,2024-05-31,2044-05-15,high,NaN,15000.0
